In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
# web scrapping
import bs4 as bs
import requests
import lxml
from functools import reduce
# matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from ipysigma import Sigma
from pyvis.network import Network
import requests
import re
from bs4 import BeautifulSoup
from io import StringIO
from dbconnection import MySQLDatabase
from utils import getSymbols, getData, get_last_date, get_marketid_simbols
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
sns.set_theme()

In [2]:
db = MySQLDatabase("financialmarkets")

In [47]:
def getSymbols(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    resp = requests.get(url, headers=headers)

    if resp.status_code != 200:
        raise ValueError(f"Error al obtener la página: {resp.status_code}")

    soup = BeautifulSoup(resp.text, "lxml")

    # Buscar cualquier tabla con clase 'wikitable'
    table = soup.find("table", {"id": lambda x: x and "constituents" in x})
    #print(len(table[]))
    #print(table)
    if table is None:
        raise ValueError("No se encontró ninguna tabla con clase 'wikitable'")

    # Intentar identificar la tabla correcta: debe contener columna con "Company" o "Firma"
    
    # Convertir tabla a DataFrame
    table = pd.read_html(StringIO(str(table)))[0]

    return table

In [57]:
df = getSymbols('https://en.wikipedia.org/wiki/DAX')
df.head()

,Logo,Company,Prime Standard Sector,Ticker,Index weighting (%)1,Employees,Founded
0,NaN,Adidas,Apparel,ADS.DE,2.0,"061,401 (2021)",1924
1,NaN,Airbus,Aerospace & Defence,AIR.PA,6.0,"126,495 (2021)",1970
2,NaN,Allianz,Financial Services,ALV.DE,7.1,"155,411 (2021)",1890
3,NaN,BASF,Chemicals,BAS.DE,3.5,"111,047 (2021)",1865
4,NaN,Bayer,Pharmaceuticals,BAYN.DE,4.8,"099,637 (2021)",1863


In [58]:
# generamos variables no existentes
#df['Símbolo'] = [re.sub(r"\s+", "", x)[:-4]+'.MX' for x in df['Símbolo'].astype('str')]
df = df.rename(columns={'Ticker':'Symbol','Company':'Security', 'Prime Standard Sector':'GICS Sector'})
df['GICS Sub-Industry'] = ['SD' for x in df['Symbol']]
df['Headquarters Location'] = ['SD' for x in df['Symbol']]
df['CIK'] = ['SD' for x in df['Symbol']]
df['Founded'] = [int(x) for x in df['Founded']]
df['Date added'] = ['0000-00-00' for x in df['Symbol']]
df = df[['Symbol','Security','GICS Sector','GICS Sub-Industry','Headquarters Location', 'Date added','CIK','Founded']]
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,ADS.DE,Adidas,Apparel,SD,SD,0000-00-00,SD,1924
1,AIR.PA,Airbus,Aerospace & Defence,SD,SD,0000-00-00,SD,1970
2,ALV.DE,Allianz,Financial Services,SD,SD,0000-00-00,SD,1890
3,BAS.DE,BASF,Chemicals,SD,SD,0000-00-00,SD,1865
4,BAYN.DE,Bayer,Pharmaceuticals,SD,SD,0000-00-00,SD,1863


**Ingresamos mercado**

In [59]:
# ---------------------------
# 3️Insertar/actualizar mercados
# ---------------------------
markets = pd.DataFrame({
    'market_name': ['DAX'],
    'country': ['GERMAN'],
    'currency': ['DEM']
})
markets

,market_name,country,currency
0,DAX,GERMAN,DEM


In [60]:
db.insert_to_db(markets, tabla="markets", batch_size=5000)

✅ Conexión exitosa


In [61]:
# Obtener market_id
market_id = db.execute_query("SELECT * FROM markets")
market_id

,market_id,market_name,country,currency
0,1,NASDAQ,USA,USD
1,2,S&P 500,USA,USD
2,3,IPC MX,MX,Peso
3,4,DAX,GERMAN,DEM


In [63]:
market_id = 4

In [64]:
df.loc[:,"market_id"] = [market_id for x in df['Symbol']]
companies = df[["market_id",'Symbol','Security','GICS Sector','GICS Sub-Industry','Date added','Headquarters Location','CIK','Founded']]
companies.columns = ["market_id",'symbol','name','sector_name','sub_industry','date_added','headquarters','cik','founded']
companies = companies.reset_index(drop=True)
companies.head()

,market_id,symbol,name,sector_name,sub_industry,date_added,headquarters,cik,founded
0,4,ADS.DE,Adidas,Apparel,SD,0000-00-00,SD,SD,1924
1,4,AIR.PA,Airbus,Aerospace & Defence,SD,0000-00-00,SD,SD,1970
2,4,ALV.DE,Allianz,Financial Services,SD,0000-00-00,SD,SD,1890
3,4,BAS.DE,BASF,Chemicals,SD,0000-00-00,SD,SD,1865
4,4,BAYN.DE,Bayer,Pharmaceuticals,SD,0000-00-00,SD,SD,1863


In [65]:
db.insert_to_db(companies, tabla="companies", batch_size=100)

In [66]:
db.close()

🔒 Conexión cerrada
